# Prédiction multiclasses du type de cancer



## 1. Imports et configuration


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, roc_auc_score, log_loss
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from scipy.stats import loguniform

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
TRAIN_SHEET = "train"
VALID_SHEET = "Validation"
TARGET = "Type"




## 2. Chargement et nettoyage


In [ ]:
def load_data(excel_path: str | Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = pd.read_excel(excel_path, sheet_name=TRAIN_SHEET)
    valid_df = pd.read_excel(excel_path, sheet_name=VALID_SHEET)
    for df in (train_df, valid_df):
        for col in df.select_dtypes(include=["object", "string"]).columns:
            df[col] = df[col].astype("string").str.strip().str.upper().replace({"NAN": np.nan, "NONE": np.nan, "": np.nan}).astype(object)
            df[col] = df[col].where(pd.notna(df[col]), np.nan)
    return train_df, valid_df


## 3. Prétraitement


In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessors(X: pd.DataFrame):
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())])
    pre_scaled = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    pre_tree = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    return pre_scaled, pre_tree


## 4. Modèles et tuning


In [ ]:
def build_searches(X: pd.DataFrame):
    pre_scaled, pre_tree = make_preprocessors(X)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    return {
        "logreg": GridSearchCV(
            Pipeline([("prep", pre_scaled), ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
            [
                {"clf__solver": ["lbfgs"], "clf__penalty": ["l2"], "clf__C": [0.1, 1, 10], "clf__class_weight": [None, "balanced"]},
                {"clf__solver": ["saga"], "clf__penalty": ["elasticnet"], "clf__C": [0.1, 1], "clf__l1_ratio": [0.5], "clf__class_weight": [None, "balanced"]},
            ],
            scoring="f1_macro", cv=cv, n_jobs=1, refit=True,
        ),
        "svm": RandomizedSearchCV(
            Pipeline([("prep", pre_scaled), ("clf", SVC(probability=True, random_state=RANDOM_STATE))]),
            {"clf__C": loguniform(1e-2, 1e2), "clf__kernel": ["linear", "rbf"], "clf__gamma": ["scale", 0.001, 0.01, 0.1], "clf__class_weight": [None, "balanced"]},
            n_iter=5, scoring="f1_macro", cv=cv, random_state=RANDOM_STATE, n_jobs=1, refit=True,
        ),
        "knn": GridSearchCV(
            Pipeline([("prep", pre_scaled), ("clf", KNeighborsClassifier())]),
            {"clf__n_neighbors": [3, 5, 7, 11, 15, 21], "clf__weights": ["uniform", "distance"], "clf__p": [1, 2]},
            scoring="f1_macro", cv=cv, n_jobs=1, refit=True,
        ),
        "random_forest": RandomizedSearchCV(
            Pipeline([("prep", pre_tree), ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1))]),
            {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 6, 10, 15], "clf__min_samples_leaf": [1, 2, 4, 8], "clf__max_features": ["sqrt", "log2", 0.8], "clf__class_weight": [None, "balanced_subsample"]},
            n_iter=6, scoring="f1_macro", cv=cv, random_state=RANDOM_STATE, n_jobs=1, refit=True,
        ),
        "gradient_boosting": RandomizedSearchCV(
            Pipeline([("prep", pre_tree), ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))]),
            {"clf__n_estimators": [50, 100, 200], "clf__learning_rate": [0.03, 0.05, 0.1], "clf__max_depth": [2, 3, 4], "clf__subsample": [0.8, 1.0], "clf__min_samples_leaf": [1, 2, 4]},
            n_iter=5, scoring="f1_macro", cv=cv, random_state=RANDOM_STATE, n_jobs=1, refit=True,
        ),
    }


## 5. Évaluation


In [ ]:
def evaluate_model(name, model, X, y_enc, le):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    pred_enc = cross_val_predict(model, X, y_enc, cv=cv, n_jobs=1, method="predict")
    try:
        proba = cross_val_predict(model, X, y_enc, cv=cv, n_jobs=1, method="predict_proba")
    except Exception:
        proba = None
    y_true = le.inverse_transform(y_enc)
    y_pred = le.inverse_transform(pred_enc)
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if proba is not None:
        row["roc_auc_ovr"] = roc_auc_score(y_enc, proba, multi_class="ovr", average="macro")
        row["log_loss"] = log_loss(y_enc, proba, labels=np.arange(len(le.classes_)))
    else:
        row["roc_auc_ovr"] = np.nan
        row["log_loss"] = np.nan
    recalls = recall_score(y_true, y_pred, average=None, labels=le.classes_, zero_division=0)
    for cls, rec in zip(le.classes_, recalls):
        row[f"recall_class_{cls}"] = rec
    return row


## 6. Chargement du fichier

Placez `donnee_etu.xlsx` dans le même dossier que ce notebook, ou modifiez `EXCEL_PATH`.


In [ ]:
EXCEL_PATH = "donnee_etu.xlsx"
OUTPUT_CSV = "predictions_tsd_cancer.csv"

train_df, valid_df = load_data(EXCEL_PATH)
X = train_df.drop(columns=[TARGET])
y_clinique = train_df[TARGET].astype(int)
X_val = valid_df.drop(columns=[TARGET], errors="ignore")

le = LabelEncoder()
y_enc = le.fit_transform(y_clinique)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Classes cliniques:", le.classes_.tolist())
print("Distribution des classes:")
print(y_clinique.value_counts().sort_index())
train_df.head()


## 7. Tuning et comparaison des modèles

La métrique d’optimisation est `f1_macro`, adaptée aux classes déséquilibrées.


In [ ]:
rows = []
fitted_searches = {}

for name, search in build_searches(X).items():
    print(f"\n=== Tuning {name} ===")
    search.fit(X, y_enc)
    fitted_searches[name] = search

    row = evaluate_model(name, clone(search.best_estimator_), X, y_enc, le)
    row["best_cv_score_f1_macro"] = search.best_score_
    row["best_params"] = search.best_params_
    rows.append(row)

    print(f"Best inner-CV F1 macro: {search.best_score_:.4f}")
    print(f"CV F1 macro estimé: {row['f1_macro']:.4f}")

results_df = pd.DataFrame(rows).sort_values(
    ["f1_macro", "balanced_accuracy", "roc_auc_ovr"],
    ascending=False,
)
results_df.to_csv("model_selection_summary.csv", index=False)
results_df[["model", "f1_macro", "balanced_accuracy", "accuracy", "roc_auc_ovr", "log_loss", "best_cv_score_f1_macro"]]


## 8. Choix, calibration et entraînement final


In [ ]:
selected_name = results_df.iloc[0]["model"]
selected_estimator = fitted_searches[selected_name].best_estimator_

try:
    final_model = CalibratedClassifierCV(estimator=clone(selected_estimator), method="sigmoid", cv=5)
except TypeError:
    final_model = CalibratedClassifierCV(base_estimator=clone(selected_estimator), method="sigmoid", cv=5)

final_model.fit(X, y_enc)
print(f"Modèle final retenu et calibré : {selected_name}")


## 9. Export du fichier demandé


In [ ]:
pred_val = le.inverse_transform(final_model.predict(X_val)).astype(int)
submission = pd.DataFrame({"Id": np.arange(len(X_val)), "Predicted_Type": pred_val})
submission.to_csv(OUTPUT_CSV, index=False)
print(f"Fichier écrit : {OUTPUT_CSV}")
print("Labels exportés :", sorted(submission["Predicted_Type"].unique().tolist()))
submission.head()


## 10. Graphiques optionnels


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cv_eval = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y_pred_enc = cross_val_predict(clone(selected_estimator), X, y_enc, cv=cv_eval, method="predict", n_jobs=1)
y_pred_clinique = le.inverse_transform(y_pred_enc)
y_true_clinique = y_clinique.to_numpy()

cm = confusion_matrix(y_true_clinique, y_pred_clinique, labels=le.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(values_format="d")
plt.title("Matrice de confusion – labels cliniques 1..6")
plt.show()


In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

try:
    y_proba = cross_val_predict(clone(selected_estimator), X, y_enc, cv=cv_eval, method="predict_proba", n_jobs=1)
    classes = le.classes_.tolist()
    y_true_bin = label_binarize(y_true_clinique, classes=classes)

    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"Classe {cls} (AUC={roc_auc:.3f})")

    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("Faux positifs")
    plt.ylabel("Vrais positifs")
    plt.title("Courbes ROC one-vs-rest")
    plt.legend()
    plt.show()
except Exception as exc:
    print("Courbes ROC non disponibles :", exc)


In [ ]:
from sklearn.calibration import CalibrationDisplay

try:
    if "y_proba" not in globals():
        y_proba = cross_val_predict(clone(selected_estimator), X, y_enc, cv=cv_eval, method="predict_proba", n_jobs=1)
    for i, cls in enumerate(le.classes_):
        y_true_binary = (y_true_clinique == cls).astype(int)
        prob_cls = y_proba[:, i]
        CalibrationDisplay.from_predictions(y_true_binary, prob_cls, n_bins=8, name=f"Classe {cls}")
        plt.title(f"Calibration OvR – classe {cls}")
        plt.show()
except Exception as exc:
    print("Graphiques de calibration non disponibles :", exc)
